# Deployment is part of the model

A trained model is not a usable artifact unless the preprocessing, label mapping, weights, and expected input shape travel with it. This notebook packages a tiny classifier, validates inputs, saves the artifact, reloads it, and exposes the prediction function that a UI or API would call.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json
import numpy as np
import torch
from torch import nn

torch.manual_seed(3)

## A packaged prediction contract

The contract makes assumptions explicit: two numeric features, standardization statistics, and two class names. In a real project these values come from training data and are saved beside the state dict.

In [ ]:
class TwoFeatureModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 2))
    def forward(self, x):
        return self.network(x)

model = TwoFeatureModel()
metadata = {
    'feature_names': ['x1', 'x2'],
    'mean': [0.0, 0.0],
    'scale': [1.0, 1.0],
    'classes': ['class_0', 'class_1'],
}
artifact = {'state_dict': model.state_dict(), 'metadata': metadata}
path = Path('two_feature_classifier.pt')
torch.save(artifact, path)
print('saved:', path.resolve())

In [ ]:
def load_artifact(path):
    artifact = torch.load(path, map_location='cpu')
    model = TwoFeatureModel()
    model.load_state_dict(artifact['state_dict'])
    model.eval()
    return model, artifact['metadata']

def predict(model, metadata, values):
    values = np.asarray(values, dtype=np.float32)
    if values.shape != (2,):
        raise ValueError(f'expected exactly two features with shape (2,), got {values.shape}')
    scale = np.asarray(metadata['scale'], dtype=np.float32)
    if np.any(scale == 0):
        raise ValueError('preprocessing scale cannot contain zero')
    normalized = (values - np.asarray(metadata['mean'], dtype=np.float32)) / scale
    with torch.no_grad():
        probabilities = model(torch.from_numpy(normalized).unsqueeze(0)).softmax(1)[0]
    index = int(probabilities.argmax())
    return {'label': metadata['classes'][index], 'probabilities': probabilities.numpy()}

loaded_model, loaded_metadata = load_artifact(path)
print(predict(loaded_model, loaded_metadata, [0.4, -0.8]))
for invalid in ([1.0], [[1.0, 2.0]]):
    try: predict(loaded_model, loaded_metadata, invalid)
    except ValueError as error: print('rejected:', error)

The saved file is a local experiment artifact, not a production service. A deployed system also needs versioned dependencies, monitoring, latency measurements, and tests for preprocessing drift. A small Gradio interface can call `predict`, but it should not duplicate its validation logic.

In [ ]:
path.unlink(missing_ok=True)
print('temporary artifact removed')